# 🧪 Fertilizer Recommendation Model Training
## Offline AI Assistant for Smart Farming
### Member 1 - AI/ML Module
---

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import confusion_matrix
from src.preprocess import preprocess_fertilizer_data
from src.fertilizer_model import FertilizerRecommendationModel

print('Libraries loaded ✅')

## Step 1: Preprocess Data

In [ ]:
X_train, X_test, y_train, y_test, scaler, encoders = preprocess_fertilizer_data()
print(f"Fertilizers: {list(encoders['Fertilizer Name'].classes_)}")

## Step 2: Train Models

In [ ]:
fert_model = FertilizerRecommendationModel()
fert_model.encoders = encoders
fert_model.scaler   = scaler
fert_model.train_random_forest(X_train, y_train)
fert_model.train_xgboost(X_train, y_train)

## Step 3: Evaluate Models

In [ ]:
results = fert_model.evaluate(X_test, y_test)

model_names = list(results.keys())
accuracies  = [results[m]['accuracy'] * 100 for m in model_names]

plt.figure(figsize=(7, 5))
bars = plt.bar(model_names, accuracies, color=['steelblue', 'coral'], edgecolor='white', width=0.5)
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 2,
             f'{acc:.2f}%', ha='center', va='top', color='white', fontweight='bold')
plt.ylim(85, 101)
plt.title('Fertilizer Model Accuracy Comparison', fontsize=14)
plt.ylabel('Accuracy (%)')
plt.tight_layout()
plt.savefig('../data/processed/fertilizer_accuracy_comparison.png', dpi=150)
plt.show()

## Step 4: Confusion Matrix

In [ ]:
best_name  = max(results, key=lambda k: results[k]['accuracy'])
best_preds = results[best_name]['predictions']
fert_classes = encoders['Fertilizer Name'].classes_

cm = confusion_matrix(y_test, best_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=fert_classes, yticklabels=fert_classes)
plt.title(f'Confusion Matrix - {best_name}', fontsize=16)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../data/processed/fertilizer_confusion_matrix.png', dpi=150)
plt.show()

## Step 5: Feature Importance

In [ ]:
feature_cols = ['Temparature', 'Humidity', 'Moisture', 'Soil Type',
                'Crop Type', 'Nitrogen', 'Potassium', 'Phosphorous']
feature_cols = [col for col in feature_cols if col in X_train.columns]
importances  = fert_model.rf_model.feature_importances_

plt.figure(figsize=(9, 5))
sorted_idx = np.argsort(importances)[::-1]
plt.bar([feature_cols[i] for i in sorted_idx],
        [importances[i] for i in sorted_idx],
        color='mediumseagreen', edgecolor='white')
plt.title('Feature Importance - Random Forest (Fertilizer Model)', fontsize=14)
plt.ylabel('Importance')
plt.xlabel('Feature')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../data/processed/fertilizer_feature_importance.png', dpi=150)
plt.show()

## Step 6: Save Models

In [ ]:
fert_model.save_models()
print('✅ All fertilizer models saved!')

## Step 7: Test Prediction

In [ ]:
test_input = {
    'Temparature': 26, 'Humidity': 52, 'Moisture': 38,
    'Soil Type': 'Sandy', 'Crop Type': 'Maize',
    'Nitrogen': 37, 'Potassium': 0, 'Phosphorous': 0
}

result = fert_model.predict(test_input)
print(f"Recommended Fertilizer : {result['recommended_fertilizer']}")
print(f"Confidence             : {result['confidence']}%")
print(f"Top 3 Fertilizers      : {result['top_3_fertilizers']}")